# Kiểm thử Suy luận Mô hình Đã Huấn luyện kết hợp RAG (1-LoRA + RAG Inference)

Thực hiện việc **đánh giá suy luận (Inference Test)** cuối cùng của mô hình **Llama-3.1-8B (1-LoRA) + RAG**.

**Quy trình suy luận:**
1. Học sinh gửi Đề bài (Prompt) và Bài viết (Essay) lên hệ thống.
2. Hệ thống tìm kiếm tương đồng trên Vector DB ChromaDB để lấy **2 bài viết tham khảo** cùng chủ đề kèm điểm chuẩn.
3. Ghép các thông tin vào Prompt Template hệ thống.
4. Mô hình 1-LoRA đã được fine-tune thực hiện đọc hiểu, chấm điểm 4 tiêu chí và xuất ra chuỗi JSON duy nhất.
5. Backend thực hiện parse JSON để lấy điểm số và hiển thị biểu đồ nhận xét.

In [1]:
import os
from dotenv import load_dotenv

# Nạp các biến môi trường cấu hình cache trước khi load Unsloth/Transformers
load_dotenv(os.path.abspath("../.env"))

import sys
import json
from unsloth import FastLanguageModel

# Thêm đường dẫn src/ để sử dụng rag_utils
sys.path.append(os.path.abspath("../src"))
from rag import rag_utils

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Nạp Mô hình đã Fine-tuned & Kích hoạt chế độ suy luận nhanh

Nạp trực tiếp mô hình nền lượng hóa kết hợp với LoRA Adapter đã lưu trong thư mục `adapters/`.

In [2]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

ADAPTER_DIR = "../adapters/llama_8b_1lora_aes"

print(f"Đang tải mô hình từ: {ADAPTER_DIR}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Kích hoạt chế độ suy luận nhanh của Unsloth (tăng tốc gấp 2 lần)
FastLanguageModel.for_inference(model)

Đang tải mô hình từ: ../adapters/llama_8b_1lora_aes...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 51.50it/s]
Unsloth: Will load ../adapters/llama_8b_1lora_aes as a legacy tokenizer.
Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

## 2. Nạp Cơ sở dữ liệu Vector RAG

In [3]:
VECTOR_DB_DIR = "../data/processed/chroma_db"
vectordb = rag_utils.load_vector_db(VECTOR_DB_DIR)
print("✔ Đã tải ChromaDB thành công.")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5111.98it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
t:\5 - Summer 2026\AES_LLM\src\rag\rag_utils.py:28: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


✔ Đã tải ChromaDB thành công.


## 3. Xây dựng Pipeline chấm điểm IELTS hoàn chỉnh

Tích hợp RAG và Model Generation vào một hàm duy nhất.

In [4]:
IELTS_EVAL_PROMPT_TEMPLATE = """You are a highly experienced IELTS writing examiner. Your goal is to provide a precise and consistent evaluation of an essay by following a structured reasoning process.

**CONTEXT (Reference Essays with Scores):**
{context}

**NEW ESSAY TO GRADE:**
{question}

**EVALUATION PROCESS (Think step-by-step):**
1. Task Response (TR) Analysis: Assess how well the 'NEW ESSAY' addresses the prompt. Compare its quality to the TR scores in the 'CONTEXT'.
2. Coherence and Cohesion (CC) Analysis: Assess structure, paragraphing, and linking. Compare to CC scores in the 'CONTEXT'.
3. Lexical Resource (LR) Analysis: Assess range and accuracy of vocabulary. Compare to LR scores in the 'CONTEXT'.
4. Grammatical Range and Accuracy (GRA) Analysis: Assess grammar range and accuracy. Compare to GRA scores in the 'CONTEXT'.

**FINAL OUTPUT FORMAT (Strict JSON):**
Your entire response MUST be a single valid JSON object containing exactly these fields. Do NOT include markdown code blocks or explanations outside JSON.
{{
  "Task_Response": {{
    "Band": <score>,
    "Comment": "<brief justification>"
  }},
  "Coherence_and_Cohesion": {{
    "Band": <score>,
    "Comment": "<brief justification>"
  }},
  "Lexical_Resource": {{
    "Band": <score>,
    "Mistakes": ["<mistake1>", "<mistake2>"],
    "Corrections": ["<correction1>", "<correction2>"],
    "Comment": "<brief justification>"
  }},
  "Grammatical_Range_and_Accuracy": {{
    "Band": <score>,
    "Mistakes": ["<mistake1>", "<mistake2>"],
    "Corrections": ["<correction1>", "<correction2>"],
    "Comment": "<brief justification>"
  }},
  "General_Feedback": "<constructive feedback>"
}}

JSON Response:
"""

In [5]:
def evaluate_essay(essay_prompt, essay_text):
    # 1. Truy xuất RAG
    retrieved_docs = rag_utils.retrieve_examples(vectordb, essay_text, k=2)
    context_str = rag_utils.format_rag_context(retrieved_docs)
    
    # 2. Xây dựng Prompt mẫu
    final_prompt = rag_utils.format_evaluation_prompt(
        prompt_template=IELTS_EVAL_PROMPT_TEMPLATE,
        context=context_str,
        essay_prompt=essay_prompt,
        essay_text=essay_text
    )
    
    # 3. Tokenize và Sinh phản hồi
    inputs = tokenizer([final_prompt], return_tensors="pt").to("cuda")

     # Lấy các token kết thúc (EOS và EOT) để ép dừng sinh
    eos_ids = [tokenizer.eos_token_id]
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if eot_id is not None and eot_id != tokenizer.unk_token_id:
        eos_ids.append(eot_id)
    
    print("⌛ Đang sinh kết quả chấm điểm từ Llama-3.1-8B (1-LoRA)...\n")
    outputs = model.generate(
        **inputs,
        max_new_tokens=768,
        use_cache=True,
        do_sample=True,
        temperature=0.1,    # Nhiệt độ thấp giúp JSON sinh ra nhất quán và ổn định hơn
        top_p=0.9,
        repetition_penalty=1.15,
        eos_token_id=eos_ids
    )
    
    # 4. Giải mã kết quả
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response_str = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    return response_str

## 4. Chạy kiểm thử trên bài viết thực tế

Gửi một bài viết mẫu và thực hiện parse JSON kết quả.

In [6]:
import json, re      

# 1. Định nghĩa prompt & essay
test_prompt = "Some people think that universities should provide graduates with the knowledge and skills needed in the workplace. Discuss both views."
test_essay = """
Nowadays, it is true that finding a job has become very competitive. Therefore, many people support the idea that universities should focus on vocational training. From my perspective, while providing professional skills is highly necessary, teaching theoretical knowledge for its own sake is also a fundamental responsibility of academic institutions.

On the one hand, specialized training directly prepares students for their future careers. For instance, engineering or nursing students cannot perform their duties without hands-on lab sessions and internship experiences. In today's market, employers value candidates who can contribute immediately without requiring expensive training programs. Thus, vocational modules help increase the employment rate among new graduates.

On the other hand, theoretical sciences foster critical thinking and problem-solving abilities. Pure academic disciplines like mathematics, philosophy, or history expand students' cognitive horizons, allowing them to comprehend complex global issues. If colleges only teach practical skills, they will become trade schools, and society might lose long-term scientific progress.

In conclusion, universities must strike a balance. They should offer courses that teach both workplace skills and theoretical academic knowledge to create well-rounded graduates.
"""

# 2. Hàm phụ trợ: Trích xuất khối JSON đầu tiên
def extract_first_json(text: str) -> str:
    """Trả về chuỗi JSON đầu tiên được bao bởi cặp {} trong `text`."""
    first_brace = text.find('{')
    if first_brace == -1:
        return text
    cnt = 0
    for i in range(first_brace, len(text)):
        if text[i] == '{':
            cnt += 1
        elif text[i] == '}':
            cnt -= 1
            if cnt == 0:
                return text[first_brace : i + 1]
    # Không tìm được cặp đóng chính xác → trả toàn bộ
    return text

# 3. Gửi essay tới mô hình & in kết quả thô
response = evaluate_essay(test_prompt, test_essay)
print("=== PHẢN HỒI THÔ TỪ MODEL ===")
print(response)                             # xem nguyên bản (có thể chứa “Improved Output…”)
print("\n" + "="*60 + "\n")

# 4. Trích xuất & parse JSON
try:
    # Lấy khối JSON duy nhất
    json_str = extract_first_json(response)
    # Chuyển sang dict Python
    json_data = json.loads(json_str)
    print("✔ ĐÃ PARSE JSON THÀNH CÔNG:")
    print(json.dumps(json_data, indent=2, ensure_ascii=False))
    
except Exception as e:
    print("❌ LỖI PARSE JSON:", e)
    print("Mẹo khắc phục:  Kiểm tra lại prompt, giảm temperature, hoặc đảm bảo mô hình được dừng bằng `eos_token_id`.")


Both `max_new_tokens` (=768) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⌛ Đang sinh kết quả chấm điểm từ Llama-3.1-8B (1-LoRA)...



t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


=== PHẢN HỒI THÔ TỪ MODEL ===
{
  "Task_Response": {
    "Band": 5,
    "Comment": "The writer provides a clear argument supporting the view that universities should prioritize vocational training."
  },
  "Coherence_and_Cohesion": {
    "Band": 6,
    "Comment": "Paragraphing is good overall, though there could be some improvement in linking between paragraphs."
  },
  "Lexical_Resource": {
    "Band": 7,
    "Mistakes": [
      "Some people think that universities should provide graduates with the knowledge and skills needed in the workplace.",
      "While providing professional skills is highly necessary, teaching theoretical knowledge for its own sake is also a fundamental responsibility of academic institutions."
    ],
    "Corrections": [
      "Nations should spend more money on skills and vocational training for practical work, rather than on university education. To what extent do you agree or disagree?",
      "It is often argued that a country should invest more money in v